In [1]:
# =============================================================================
# 02_actionability_constrained_dice.ipynb
# =============================================================================
# Purpose:
#   Compare UNCONSTRAINED DiCE counterfactuals (features_to_vary = all 30) with
#   ACTIONABILITY-CONSTRAINED counterfactuals (features_to_vary = 19 owner-
#   controllable "sales" features only) on a matched sample of closed-restaurant
#   instances. For each regime we evaluate the four standard DiCE metrics
#   (validity, proximity, sparsity, diversity), a new Actionability Rate, and the
#   paper's standardized composite score across 2/4/6/8 counterfactuals. We also
#   run a weight-sensitivity analysis of the composite score.
#
# Notes:
#   * Model is retrained in-notebook (XGBoost on the 30 SHAP features) so the
#     experiment is fully reproducible without loading pickled objects.
#   * DiCE's post-hoc sparsity optimisation is disabled (posthoc_sparsity_param
#     =None) for tractable runtime; sparsity is measured directly from the
#     generated counterfactuals. A fixed random sample keeps the run bounded.
#
# Inputs:
#   ../data/modeling_dataset_top30.csv, ../data/selected_features.pkl
# Outputs (results/):
#   tables/t3_dice_metrics_by_regime.csv
#   tables/t4_composite_score_by_regime.csv
#   tables/t5_weight_sensitivity.csv
#   figures/f4_metric_comparison.(png|pdf)
#   figures/f5_actionability_rate.(png|pdf)
#   figures/f6_composite_score.(png|pdf)
# =============================================================================

import warnings, time
from pathlib import Path
from itertools import combinations
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from scipy import stats as sps
import xgboost as xgb
import dice_ml

warnings.filterwarnings("ignore")

# ----------------------------------------------------------------------------
# Plot style: grayscale, no titles/captions, 600 dpi, png + pdf
# ----------------------------------------------------------------------------
sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({"font.size": 11, "axes.edgecolor": "black", "axes.linewidth": 0.8,
                     "savefig.dpi": 600, "figure.dpi": 120, "pdf.fonttype": 42, "ps.fonttype": 42})
GRAY_DARK, GRAY_MID, GRAY_LIGHT = "0.20", "0.45", "0.75"
FIG_DIR = Path("../results/figures"); FIG_DIR.mkdir(parents=True, exist_ok=True)
TAB_DIR = Path("../results/tables");  TAB_DIR.mkdir(parents=True, exist_ok=True)
def save_fig(fig, name):
    fig.tight_layout()
    fig.savefig(FIG_DIR / f"{name}.png", dpi=600, bbox_inches="tight")
    fig.savefig(FIG_DIR / f"{name}.pdf", bbox_inches="tight")
    plt.close(fig); print(f"  saved figure: {name}.png / {name}.pdf")

# ----------------------------------------------------------------------------
# Experiment configuration
# ----------------------------------------------------------------------------
SEED = 44
THRESHOLD = 0.3142              # classification threshold from the original study
N_SAMPLE = 120                  # matched sample of closed-restaurant instances
CF_SETTINGS = [2, 4, 6, 8]      # number of counterfactuals per instance
WEIGHTS = {"validity": 0.4, "proximity": 0.3, "sparsity": 0.2, "diversity": 0.1}

# ----------------------------------------------------------------------------
# PART 1: Data, taxonomy, and model
# ----------------------------------------------------------------------------
print("=" * 78); print("PART 1: DATA, TAXONOMY, MODEL"); print("=" * 78)
features = joblib.load("../data/selected_features.pkl")
df = pd.read_csv("../data/modeling_dataset_top30.csv")
ACTIONABLE = [f for f in features if "Sales" in f]
IMMUTABLE  = [f for f in features if "Population" in f]
print(f"  actionable (sales) = {len(ACTIONABLE)} | immutable (population) = {len(IMMUTABLE)}")

X, y = df[features], df["closure"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
model = xgb.XGBClassifier(learning_rate=0.01, max_depth=5, min_child_weight=3,
                          n_estimators=100, subsample=0.8, random_state=SEED,
                          eval_metric="logloss")
model.fit(X_tr, y_tr)
auc = roc_auc_score(y_te, model.predict_proba(X_te)[:, 1])
print(f"  XGBoost(top30) test AUC = {auc:.4f}  (predictive ceiling is low by design)")

# MAD per feature for proximity normalisation (computed on full data).
mad = {f: (max((df[f] - df[f].median()).abs().median(), 1.0)) for f in features}

# ----------------------------------------------------------------------------
# PART 2: DiCE set-up and matched closed-store sample
# ----------------------------------------------------------------------------
print("\n" + "=" * 78); print("PART 2: DiCE SET-UP + MATCHED SAMPLE"); print("=" * 78)
dice_data = dice_ml.Data(dataframe=df[features + ["closure"]].copy(),
                         continuous_features=features, outcome_name="closure")
dice_model = dice_ml.Model(model=model, backend="sklearn")
explainer = dice_ml.Dice(dice_data, dice_model, method="random")

# Query set = closed restaurants that the model also PREDICTS to close at the
# native 0.5 boundary. For these, DiCE's "opposite" target is unambiguously the
# operating class, so the Closure -> Operating counterfactuals are well posed.
proba_all = model.predict_proba(df[features])[:, 1]
candidates = df.index[(df["closure"] == 1) & (proba_all >= 0.5)].tolist()
rng = np.random.RandomState(SEED)
sample_idx = sorted(rng.choice(candidates, size=min(N_SAMPLE, len(candidates)),
                               replace=False).tolist())
print(f"  closed & predicted-to-close restaurants: {len(candidates)} | matched sample: {len(sample_idx)}")

# ----------------------------------------------------------------------------
# PART 3: Counterfactual generation (unconstrained vs. actionability-constrained)
# ----------------------------------------------------------------------------
print("\n" + "=" * 78); print("PART 3: COUNTERFACTUAL GENERATION"); print("=" * 78)

GEN_LOG = []   # tidy record of every generated counterfactual (for reproducibility / NB3)

def generate(regime, total_cfs):
    """Return a list of per-store dicts with original row and its CF rows."""
    ftv = ACTIONABLE if regime == "constrained" else features
    out = []
    for i in sample_idx:
        q = df[features].iloc[i:i + 1]
        store_id = df["store_id"].iloc[i]
        try:
            res = explainer.generate_counterfactuals(
                q, total_CFs=total_cfs, desired_class="opposite",
                features_to_vary=ftv, random_seed=SEED, verbose=False,
                posthoc_sparsity_param=None)
            cfdf = res.cf_examples_list[0].final_cfs_df
            if cfdf is not None and len(cfdf) > 0:
                cfv = cfdf[features].reset_index(drop=True)
                out.append({"store_id": store_id, "orig": q.iloc[0], "cfs": cfv})
                rec = cfv.copy()
                rec.insert(0, "store_id", store_id)
                rec.insert(1, "regime", regime); rec.insert(2, "num_cfs", total_cfs)
                GEN_LOG.append(rec)
        except Exception:
            pass
    return out

ACT_IDX = [features.index(f) for f in ACTIONABLE]

def per_store_metrics(pack):
    """Per-store metrics for one pack. Validity is reported at BOTH the native
    0.5 boundary and the study's operational closure threshold (0.3142); the
    latter is the decision policy under which 'operating' advice is acted upon."""
    o, C = pack["orig"], pack["cfs"]
    probas = model.predict_proba(C.values)[:, 1]
    val05 = float(np.mean((probas < 0.5)))
    val_op = float(np.mean((probas < THRESHOLD)))      # operational flip-rate
    dists, sp, a_changes, tot_changes = [], [], 0, 0
    for _, c in C.iterrows():
        diff = np.abs(o.values - c.values)
        dists.append(np.sum([diff[k] / mad[features[k]] for k in range(len(features))]))
        changed = diff > 1e-6
        sp.append(np.mean(~changed))
        tot_changes += int(changed.sum())
        a_changes += int(changed[ACT_IDX].sum())
    if len(C) > 1:
        div = np.mean([np.mean(C.iloc[a].values != C.iloc[b].values)
                       for a, b in combinations(range(len(C)), 2)])
    else:
        div = 0.0
    return dict(store_id=pack["store_id"], validity=val05, validity_op=val_op,
                proximity=-np.mean(dists), sparsity=np.mean(sp), diversity=div,
                actionability=(a_changes / tot_changes if tot_changes else 1.0))

def metrics(packs):
    ps = pd.DataFrame([per_store_metrics(p) for p in packs])
    agg = {c: ps[c].mean() for c in ["validity", "validity_op", "proximity",
                                     "sparsity", "diversity", "actionability"]}
    agg["n"] = len(ps)
    return agg, ps

records = []
PACKS = {}          # (regime, ncf) -> list of packs, reused for statistics
PERSTORE = {}       # (regime, ncf) -> per-store DataFrame
t0 = time.time()
for regime in ["unconstrained", "constrained"]:
    for ncf in CF_SETTINGS:
        packs = generate(regime, ncf)
        PACKS[(regime, ncf)] = packs
        m, ps = metrics(packs); PERSTORE[(regime, ncf)] = ps
        m.update(regime=regime, num_cfs=ncf)
        records.append(m)
        print(f"  {regime:<13} {ncf} CFs: n={m['n']:>3}  "
              f"valid={m['validity']:.3f} valid_op={m['validity_op']:.3f} "
              f"prox={m['proximity']:.2f} spars={m['sparsity']:.3f} "
              f"div={m['diversity']:.3f} action={m['actionability']:.3f}")
print(f"  (generation time: {time.time()-t0:.0f}s)")

res = pd.DataFrame(records)
res.to_csv(TAB_DIR / "t3_dice_metrics_by_regime.csv", index=False)
if GEN_LOG:
    pd.concat(GEN_LOG, ignore_index=True).to_csv(
        TAB_DIR / "cf_generated_sample.csv", index=False, encoding="utf-8-sig")
    print(f"  saved generated counterfactuals: cf_generated_sample.csv")

# ----------------------------------------------------------------------------
# PART 4: Standardized composite score (paper methodology) per regime
# ----------------------------------------------------------------------------
print("\n" + "=" * 78); print("PART 4: STANDARDIZED COMPOSITE SCORE"); print("=" * 78)

def composite(group, weights):
    g = group.copy()
    # min-max normalise proximity (negative distance), then z-score all four.
    pr = g["proximity"]
    g["prox_n"] = 0.0 if pr.max() == pr.min() else (pr - pr.min()) / (pr.max() - pr.min())
    for raw, col in [("validity", "validity"), ("proximity", "prox_n"),
                     ("sparsity", "sparsity"), ("diversity", "diversity")]:
        s = g[col].std(ddof=1)
        g[f"{raw}_z"] = 0.0 if (s == 0 or pd.isna(s)) else (g[col] - g[col].mean()) / s
    g["total_score"] = sum(weights[k] * g[f"{k}_z"] for k in weights)
    return g

comp_all = []
for regime, grp in res.groupby("regime"):
    comp_all.append(composite(grp.sort_values("num_cfs"), WEIGHTS))
comp = pd.concat(comp_all, ignore_index=True)
comp_out = comp[["regime", "num_cfs", "validity_z", "proximity_z", "sparsity_z",
                 "diversity_z", "total_score", "actionability"]]
comp_out.to_csv(TAB_DIR / "t4_composite_score_by_regime.csv", index=False)
print(comp_out.round(4).to_string(index=False))
for regime, grp in comp.groupby("regime"):
    best = grp.loc[grp["total_score"].idxmax()]
    print(f"  optimal #CF ({regime}): {int(best['num_cfs'])} "
          f"(total score {best['total_score']:.4f}, actionability {best['actionability']:.3f})")

# ----------------------------------------------------------------------------
# PART 5: Weight-sensitivity of the composite score (robustness of optimal #CF)
# ----------------------------------------------------------------------------
print("\n" + "=" * 78); print("PART 5: WEIGHT SENSITIVITY"); print("=" * 78)
grid = []
for wv in [0.25, 0.40, 0.55]:            # weight on validity
    for wp in [0.15, 0.30, 0.45]:        # weight on proximity
        rem = 1.0 - wv - wp
        if rem <= 0:
            continue
        ws = {"validity": wv, "proximity": wp, "sparsity": rem * 2/3, "diversity": rem * 1/3}
        for regime, grp in res.groupby("regime"):
            g = composite(grp.sort_values("num_cfs"), ws)
            best = g.loc[g["total_score"].idxmax()]
            grid.append({"regime": regime, "w_validity": wv, "w_proximity": wp,
                         "optimal_num_cfs": int(best["num_cfs"])})
wsens = pd.DataFrame(grid)
wsens.to_csv(TAB_DIR / "t5_weight_sensitivity.csv", index=False)
for regime, grp in wsens.groupby("regime"):
    vc = grp["optimal_num_cfs"].value_counts().to_dict()
    print(f"  {regime}: optimal #CF across {len(grp)} weightings -> {vc}")

# ----------------------------------------------------------------------------
# PART 5b: Matched pairwise statistics (bootstrap CIs + paired tests)
# ----------------------------------------------------------------------------
print("\n" + "=" * 78); print("PART 5b: MATCHED PAIRWISE STATISTICS"); print("=" * 78)
STAT_NCF = 2        # report statistics at the parsimonious optimum
B = 2000            # bootstrap resamples
rngb = np.random.RandomState(SEED)

def boot_ci(x, fn=np.mean, b=B, alpha=0.05):
    x = np.asarray(x, float)
    stats = [fn(x[rngb.randint(0, len(x), len(x))]) for _ in range(b)]
    return float(fn(x)), float(np.percentile(stats, 100*alpha/2)), float(np.percentile(stats, 100*(1-alpha/2)))

u_ps = PERSTORE[("unconstrained", STAT_NCF)].set_index("store_id")
c_ps = PERSTORE[("constrained",  STAT_NCF)].set_index("store_id")
matched = sorted(set(u_ps.index) & set(c_ps.index))
print(f"  matched stores present in both regimes at {STAT_NCF} CFs: {len(matched)}")

rows = []
for met in ["proximity", "sparsity", "actionability", "diversity", "validity_op"]:
    u = u_ps.loc[matched, met].values
    c = c_ps.loc[matched, met].values
    um, ul, uh = boot_ci(u); cm, cl, ch = boot_ci(c)
    d = c - u
    dm, dl, dh = boot_ci(d)
    # paired Wilcoxon signed-rank (two-sided); guard the all-zero-difference case
    if np.allclose(d, 0):
        W, pval = np.nan, 1.0
    else:
        W, pval = sps.wilcoxon(c, u, zero_method="wilcox", alternative="two-sided")
    rows.append({"metric": met,
                 "unconstrained_mean": um, "unconstrained_ci": f"[{ul:.3f}, {uh:.3f}]",
                 "constrained_mean": cm, "constrained_ci": f"[{cl:.3f}, {ch:.3f}]",
                 "paired_diff(C-U)": dm, "diff_ci": f"[{dl:.3f}, {dh:.3f}]",
                 "wilcoxon_p": pval})
stat_tab = pd.DataFrame(rows)
stat_tab.to_csv(TAB_DIR / "t6_matched_pairwise_stats.csv", index=False)
pd.set_option("display.width", 200)
print(stat_tab.to_string(index=False,
      formatters={"unconstrained_mean": "{:.3f}".format, "constrained_mean": "{:.3f}".format,
                  "paired_diff(C-U)": "{:+.3f}".format, "wilcoxon_p": "{:.2e}".format}))
print("  (95% bootstrap CIs over matched stores; two-sided paired Wilcoxon signed-rank)")

# ----------------------------------------------------------------------------
# PART 6: Figures
# ----------------------------------------------------------------------------
print("\n" + "=" * 78); print("PART 6: FIGURES"); print("=" * 78)
piv = res.pivot(index="num_cfs", columns="regime")

# F4: four raw metrics vs #CF, unconstrained vs constrained.
fig, axes = plt.subplots(2, 2, figsize=(8.6, 6.6))
for ax, met, lab in zip(axes.ravel(), ["validity", "proximity", "sparsity", "diversity"],
                        ["Validity", "Proximity", "Sparsity", "Diversity"]):
    ax.plot(piv.index, piv[(met, "unconstrained")], marker="o", color=GRAY_DARK,
            linestyle="-", label="Unconstrained")
    ax.plot(piv.index, piv[(met, "constrained")], marker="s", color=GRAY_MID,
            linestyle="--", label="Constrained")
    ax.set_xlabel("Number of counterfactuals"); ax.set_ylabel(lab)
    ax.set_xticks(CF_SETTINGS)
axes.ravel()[0].legend(frameon=True, loc="best")
save_fig(fig, "f4_metric_comparison")

# F5: actionability rate vs #CF.
fig, ax = plt.subplots(figsize=(6.4, 4.2))
ax.plot(piv.index, piv[("actionability", "unconstrained")], marker="o",
        color=GRAY_DARK, linestyle="-", label="Unconstrained")
ax.plot(piv.index, piv[("actionability", "constrained")], marker="s",
        color=GRAY_MID, linestyle="--", label="Constrained")
ax.set_xlabel("Number of counterfactuals"); ax.set_ylabel("Actionability rate")
ax.set_ylim(0, 1.05); ax.set_xticks(CF_SETTINGS); ax.legend(frameon=True, loc="best")
save_fig(fig, "f5_actionability_rate")

# F6: standardized composite score vs #CF.
fig, ax = plt.subplots(figsize=(6.4, 4.2))
for regime, mk, ls, col in [("unconstrained", "o", "-", GRAY_DARK),
                            ("constrained", "s", "--", GRAY_MID)]:
    g = comp[comp["regime"] == regime].sort_values("num_cfs")
    ax.plot(g["num_cfs"], g["total_score"], marker=mk, linestyle=ls, color=col,
            label=regime.capitalize())
ax.set_xlabel("Number of counterfactuals"); ax.set_ylabel("Composite score (standardized)")
ax.set_xticks(CF_SETTINGS); ax.legend(frameon=True, loc="best")
save_fig(fig, "f6_composite_score")

print("\n[NB2 DONE] constrained vs. unconstrained DiCE experiment complete.")


PART 1: DATA, TAXONOMY, MODEL
  actionable (sales) = 19 | immutable (population) = 11


  XGBoost(top30) test AUC = 0.6442  (predictive ceiling is low by design)

PART 2: DiCE SET-UP + MATCHED SAMPLE
  closed & predicted-to-close restaurants: 164 | matched sample: 120

PART 3: COUNTERFACTUAL GENERATION


  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:00<00:00,  3.94it/s]

100%|██████████| 1/1 [00:00<00:00,  3.93it/s]

...

  constrained: optimal #CF across 8 weightings -> {2: 8}
  unconstrained: optimal #CF across 8 weightings -> {6: 5, 8: 3}

PART 5b: MATCHED PAIRWISE STATISTICS
  matched stores present in both regimes at 2 CFs: 120


       metric unconstrained_mean     unconstrained_ci constrained_mean     constrained_ci paired_diff(C-U)            diff_ci wilcoxon_p
    proximity           -163.032 [-188.292, -137.801]          -21.929 [-24.335, -19.523]         +141.103 [113.752, 167.729]   1.74e-12
     sparsity              0.935       [0.934, 0.936]            0.947     [0.945, 0.948]           +0.011     [0.010, 0.013]   3.42e-17
actionability              0.532       [0.499, 0.566]            1.000     [1.000, 1.000]           +0.468     [0.435, 0.501]   1.62e-21
    diversity              0.102       [0.101, 0.103]            0.101     [0.095, 0.106]           -0.001    [-0.007, 0.005]   3.03e-01
  validity_op              0.046       [0.021, 0.075]            0.067     [0.037, 0.100]           +0.021    [-0.021, 0.062]   3.36e-01
  (95% bootstrap CIs over matched stores; two-sided paired Wilcoxon signed-rank)

PART 6: FIGURES


  saved figure: f4_metric_comparison.png / f4_metric_comparison.pdf


  saved figure: f5_actionability_rate.png / f5_actionability_rate.pdf


  saved figure: f6_composite_score.png / f6_composite_score.pdf

[NB2 DONE] constrained vs. unconstrained DiCE experiment complete.
